# **GENETIC ALGORITHM**

This is a genetic algorithm implementation from scratch 

First, we need to see what is the correct solution for our problem, so we can compare it with the solution we get from our genetic algorithm. In this case, we wiil use the `linear regression` of scikit-learn to find the correct solution for our problem.

In [1]:
import random
import numpy as np
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


First, load the `.csv` file.

In [3]:
df = pd.read_csv("/content/drive/MyDrive/Student_Performance.csv")

In previous notebooks, we have seen that we can transform the `Extracurricular Activities` column into a binary column, where `1` means that the student has extracurricular activities and `0` means that the student does not have extracurricular activities. Perhaps, this column does not have a significant impact on the final grade of the student, so we can drop it from our dataset.

In [4]:
df = df.drop("Extracurricular Activities", axis=1)

Now, we need to split our dataset into X and y, where X is the features and y is the target variable. In this case, our target variable is the `Performance Index` column.

In [5]:
X = df.drop('Performance Index', axis=1)
Y = df['Performance Index']

## **LINEAR REGRESSION**

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.35, random_state=42)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
lin_reg.intercept_, lin_reg.coef_

(np.float64(-33.47928062671813),
 array([2.85952128, 1.01571133, 0.4706929 , 0.18406357]))

## **GENETIC ALGORITHM**

In [7]:
def initialize_poblation(size=100):
    random_seed = 42
    np.random.seed(random_seed)
    random.seed(42)
    poblacion = np.random.uniform(-3,3, size=(size, 4))
    poblacion = pd.DataFrame(poblacion)
    poblacion = poblacion.rename(columns={0: "theta_0", 1: "theta_1", 2: "theta_2", 3: "theta_3"})
    poblacion["error"] = 0
    return poblacion

In [8]:
def fitness( individuos: pd.DataFrame, X: pd.DataFrame, Y: pd.Series ):
    X = X.astype(float)
    Y = Y.astype(float).values
    resultados_error = []
    individuos = individuos.drop('error', axis=1)

    for _, individuo in individuos.iterrows():

        # Extraemos los genes
        theta = individuo.values.astype(float)

        y_pred = X.values @ theta
        error = np.mean((Y - y_pred) ** 2)
    
        resultados_error.append(error)
    
    error = np.array(resultados_error)
    individuos = individuos.copy()
    individuos["error"] = error

    return individuos


In [9]:
def best_50(poblacion: pd.DataFrame):
    ordered_poblation = poblacion.sort_values(by="error", ascending=True)
    top50= ordered_poblation.head(50)
    return top50

In [10]:
def cruce(individuos: pd.DataFrame, gamma: float = 0.6):
  poblacion = individuos.copy()
  theta_0_values = individuos["theta_0"].values.flatten()
  theta_1_values = individuos["theta_1"].values.flatten()
  theta_2_values = individuos["theta_2"].values.flatten()
  theta_3_values = individuos["theta_3"].values.flatten()

  for i in range(0, len(theta_1_values),2):
    if random.random() <= gamma:
      theta_0_values[i], theta_0_values[i + 1] = theta_0_values[i + 1], theta_0_values[i]
      theta_2_values[i], theta_2_values[i + 1] = theta_2_values[i + 1], theta_2_values[i]
    else:
      theta_1_values[i], theta_1_values[i + 1] = theta_1_values[i + 1], theta_1_values[i]
      theta_3_values[i], theta_3_values[i + 1] = theta_3_values[i + 1], theta_3_values[i]

  poblacion["theta_0"] = theta_0_values
  poblacion["theta_1"] = theta_1_values
  poblacion["theta_2"] = theta_2_values
  poblacion["theta_3"] = theta_3_values

  nueva_poblacion = pd.concat([individuos, poblacion], ignore_index = False)
  nueva_poblacion["error"] = 0

  return nueva_poblacion

In [11]:
def mutacion(poblacion: pd.DataFrame, prob_mutacion: float = 0.65):
    poblacion = poblacion.drop('error', axis=1)
    for idx,_ in poblacion.iterrows():
        for col in poblacion.columns:
            if random.random() <= prob_mutacion:
                poblacion.loc[idx, col] += random.uniform(-0.001, 0.001)
    
    poblacion["error"] = 0
    return poblacion

In [12]:
def genetic_algorithm(generations: int, X: pd.DataFrame, Y: pd.Series, poblation_size: int = 100):
    poblation = initialize_poblation(size=poblation_size)
    poblation = fitness(
        X = X,
        Y = Y,
        individuos = poblation
    )

    for i in range(generations):
        top50 = best_50(poblacion= poblation)

        new_generation = cruce( 
            individuos= top50 
        )

        new_generation = mutacion(
            poblacion = new_generation
        )

        poblation = new_generation

        poblation = fitness(
            X = X,
            Y = Y,
            individuos = poblation
        )
        
        # print(f'[INFO] Generación {i+1}')
    
    best_gen = poblation.sort_values(by='error').iloc[0]

    return best_gen

In [13]:
gen50  = genetic_algorithm(
    generations = 50,
    X = X,
    Y = Y
)

gen50

,87
theta_0,2.190153
theta_1,0.820051
theta_2,-1.040885
theta_3,-0.958420
error,39.183014


In [14]:
gen100  = genetic_algorithm(
    generations = 100,
    X = X,
    Y = Y
)

gen100

,87
theta_0,2.204062
theta_1,0.820729
theta_2,-1.017885
theta_3,-0.894283
error,39.530593


In [15]:
gen500  = genetic_algorithm(
    generations = 500,
    X = X,
    Y = Y
)

gen500

,87
theta_0,2.187367
theta_1,0.871203
theta_2,-1.119232
theta_3,-0.763690
error,57.986550


In [16]:
gen1000  = genetic_algorithm(
    generations = 1000,
    X = X,
    Y = Y
)

gen1000

,87
theta_0,2.220870
theta_1,0.814576
theta_2,-1.117681
theta_3,-0.730548
error,37.321847


In [17]:
predict = lin_reg.predict(X)
print(predict)

[91.5130895  63.49803213 44.86114482 ... 72.66774834 94.07526004
 65.64961341]


In [18]:
X = X.to_numpy()

In [22]:
lin_reg.predict([X[2]])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([44.86114482])

In [20]:
gen100 = gen100.drop('error')
gen500 = gen500.drop('error')
gen1000 = gen1000.drop('error')

In [23]:
print(X[2] @ gen100)
print(X[2] @ gen500)
print(X[2] @ gen1000)

50.57589295945225
52.568305863492675
50.0254690504501


In [24]:
gen1000.to_numpy()

array([ 2.22087026,  0.81457585, -1.11768073, -0.73054816])

## **METRICS**

In this case, to evaluate our model, we need to create a `predict` function that takes the features and makes a prediction of the target variable.

In [25]:
def predict(X):
    gen = [ 2.22087026,  0.81457585, -1.11768073, -0.73054816]
    return X @ gen

In [26]:
y_pred_genetic = predict(X_test)
y_pred_genetic = y_pred_genetic.to_numpy()

In [27]:
y_pred = lin_reg.predict(X_test)

Let's calculate the following metrics:
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- R-squared (R2)

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [29]:
mae_genetic_algorithm = mean_absolute_error(y_test, y_pred_genetic)
mse_genetic_algorithm = mean_squared_error(y_test, y_pred_genetic)
r2_genetic_algorithm = r2_score(y_test, y_pred_genetic)

print(f'Genetic Algorithm - MAE: {mae_genetic_algorithm}')
print(f'Genetic Algorithm - MSE: {mse_genetic_algorithm}')
print(f'Genetic Algorithm - R2 Score: {r2_genetic_algorithm}')

Genetic Algorithm - MAE: 5.01966761438857
Genetic Algorithm - MSE: 39.05043776562626
Genetic Algorithm - R2 Score: 0.8950157297206219


In [30]:
mae_linear_regression = mean_absolute_error(y_test, y_pred)
mse_linear_regression = mean_squared_error(y_test, y_pred)
r2_linear_regression = r2_score(y_test, y_pred)

print(f'Genetic Algorithm - MAE: {mae_linear_regression}')
print(f'Genetic Algorithm - MSE: {mse_linear_regression}')
print(f'Genetic Algorithm - R2 Score: {r2_linear_regression}')

Genetic Algorithm - MAE: 1.636774734466225
Genetic Algorithm - MSE: 4.2255969052018365
Genetic Algorithm - R2 Score: 0.9886397891298954
